[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/modes/modes.ipynb)

# Vibration Modes of a Suspended Plate

This example computes the natural vibration frequencies and normal mode shapes of a planar rigid plate suspended from elastic springs. The plate is modelled as a rigid body about an unstressed equilibrium whose small displacements mix horizontal sliding, vertical bounce, and rotation according to where the springs attach and how the mass is spread. Vertical springs alone leave horizontal sliding unrestrained at zero frequency while vertical bounce and rocking oscillate independently, whereas an additional angled spring on the side restrains the horizontal motion and couples translation with rotation in every mode.

The term ***forque*** here means a combined force and torque on a rigid body, represented as an antibivector alongside bivector twists for infinitesimal motions. Pairing a spring's normalized line of action with `Twist` via `Twist & line` reads its first-order extension, so `spring_constant * line * (Twist & line)` builds the spring's Hookean stiffness map from twists to forques while `(masses * mass_points).regressive(mass_points.commutator(Twist))` builds each mass point's inertia contribution directly in world space. Pairing both maps with a second twist yields the symmetric energy forms `Twist & stiffness` and `Twist & inertia`, and `potential_form.eigh(kinetic_form)` finds the mode twists where potential and kinetic energy are proportional.

In the matrix notation of planar Newton–Euler dynamics, generalized coordinates $\mathbf{q} = (u_x, u_y, \theta)^\top$ are referred to a chosen origin $\mathbf{r}_0 = (x_0, y_0)^\top$, and each mass $m_i$ at $\mathbf{r}_i$ enters $\mathbf{M} = \sum_i m_i \mathbf{J}_i^\top \mathbf{J}_i$ through the lever-arm Jacobian $\mathbf{J}_i = \begin{bmatrix} 1 & 0 & -(y_i - y_0) \\ 0 & 1 & x_i - x_0 \end{bmatrix}$ or a parallel-axis shift from the centre of mass. Each spring of stiffness $k_s$ and unit direction $\hat{\mathbf{n}}_s = (n_{sx}, n_{sy})^\top$ attached at $\mathbf{r}_s$ contributes $k_s \mathbf{b}_s \mathbf{b}_s^\top$ to $\mathbf{K}$ via $\mathbf{b}_s = \bigl(n_{sx},\, n_{sy},\, (x_s - x_0) n_{sy} - (y_s - y_0) n_{sx}\bigr)^\top$ before solving $\mathbf{K}\mathbf{v} = \omega^2 \mathbf{M}\mathbf{v}$.

In [ ]:
# Install numga and its example dependencies when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import PGA2D
from examples.util.animation import show_states
from examples.mechanics.modes import scenarios
from examples.mechanics.modes.render import mode_figure, mode_frames, setup_figure

ga = PGA2D
mv = NumpyContext(ga).multivector

Scalar = ga.gatype.scalar()                          # Scalar+
Point = ga.gatype.antivector()                       # Antivector-
Twist = ga.gatype.bivector()                         # Bivector+
Forque = ga.gatype.antibivector()                    # Antibivector-
Stiffness = ga.gatype((Forque, Twist))               # Forque- <- Twist+
Inertia = ga.gatype((Forque, Twist))                 # Forque- <- Twist+

## Unrestrained modes on vertical springs

A uniform rectangular plate hanging from symmetric vertical springs has exact mass, centre of mass, and rotational inertia captured by four equal point masses at its two-point Gauss quadrature nodes. Summing the momentum maps of those four points gives the plate's inertia, summing the vertical spring dyads gives its stiffness, and solving the potential energy form against the kinetic energy form separates the three planar degrees of freedom into independent modes.

In [ ]:
corners = mv.point(scenarios.plate_coordinates)                              # [4] Point-
masses = scenarios.plate_masses                                              # [4]
vertical_anchors = mv.point(scenarios.vertical_anchor_coordinates)           # [2] Point-
vertical_attachments = mv.point(scenarios.vertical_attachment_coordinates)   # [2] Point-
vertical_constants = scenarios.vertical_spring_constants                     # [2]

setup_figure(corners, vertical_anchors, vertical_attachments);

The rectangular plate hangs at equilibrium from two vertical springs anchored overhead and attached symmetrically near its upper corners.

In [ ]:
def quadrature_points(corners: Point) -> Point:
    """Two-point Gauss quadrature places four equal-weight mass nodes (`[4] Point-`) inside a rectangular plate with `corners` (`[4] Point-`)."""
    # Two-point Gauss quadrature integrates quadratic mass moments on the rectangle with four equal weights.
    centre = corners.sum(axis=0) / 4                                         # [] Point-
    offsets = corners - centre                                               # [4] Point-
    return centre + offsets / np.sqrt(3)


def body_inertia(mass_points: Point, masses: np.ndarray) -> Inertia:
    """Momentum lines across `mass_points` (`[...] Point-`) weighted by `masses` sum into the rigid-body inertia (`[] Forque- <- Twist+`)."""
    # Joining each mass-weighted point with its commutator velocity builds its momentum map without a reference point.
    velocities = mass_points.commutator(Twist)                               # [...] Point- <- Twist+
    point_momenta = (masses * mass_points).regressive(velocities)            # [...] Forque- <- Twist+
    return point_momenta.sum(axis=0)


mass_points = quadrature_points(corners)                                     # [4] Point-
inertia = body_inertia(mass_points, masses)                                  # [] Forque- <- Twist+

In [ ]:
def spring_stiffness(anchors: Point, attachments: Point, spring_constants: np.ndarray) -> Stiffness:
    """Hookean dyads of springs joining `anchors` to `attachments` (`[...] Point-`) with `spring_constants` sum into the stiffness map (`[] Forque- <- Twist+`)."""
    # Pairing a twist with each unit spring line reads its first-order extension, and multiplying back pulls along the same line.
    lines = (anchors & attachments).normalized()                             # [...] Forque-
    extensions = Twist & lines                                               # [...] Scalar+ <- Twist+
    dyads = spring_constants * lines * extensions                            # [...] Forque- <- Twist+
    return dyads.sum(axis=0)


def normal_modes(stiffness: Stiffness, inertia: Inertia) -> tuple[Scalar, Twist]:
    """Generalized eigenanalysis of the potential and kinetic energy forms yields the natural frequencies (`[3] Scalar+`, in hertz) and mass-normalized mode twists (`[3] Twist+`)."""
    # Pairing stiffness and inertia with a second twist turns both maps into symmetric bilinear energy forms.
    potential_form = Twist & stiffness                                       # [] Scalar+ <- (Twist+, Twist+)
    kinetic_form = Twist & inertia                                           # [] Scalar+ <- (Twist+, Twist+)
    eigenvalues, mode_twists = potential_form.eigh(kinetic_form)             # [3] Scalar+, Twist+
    frequencies = eigenvalues.clip(0, np.inf).square_root() / (2 * np.pi)    # [3] Scalar+
    return frequencies, mode_twists


unrestrained_stiffness = spring_stiffness(vertical_anchors, vertical_attachments, vertical_constants)  # [] Forque- <- Twist+
unrestrained_frequencies, unrestrained_modes = normal_modes(unrestrained_stiffness, inertia)           # [3] Scalar+, Twist+
print("Unrestrained frequencies (Hz):", np.round(unrestrained_frequencies.to_array(), 3))
mode_figure(corners, vertical_anchors, vertical_attachments, unrestrained_modes);

Each panel overlays a displaced mode shape on the dotted equilibrium outline, ordered from lowest to highest frequency. In the left panel the plate slides sideways at zero frequency without stretching either vertical spring to first order, in the middle panel both springs stretch together in pure vertical bounce, and in the right panel the two springs work in opposition as the plate rocks about its centre.

## Coupled modes with an angled spring

Attaching a third spring at an angle on the right edge adds its stiffness dyad directly to the two-spring stiffness map while leaving the plate's inertia unchanged. Because the angled line of action passes off the plate's centre and pulls both horizontally and vertically, the zero-frequency slide disappears and every mode combines horizontal shift, vertical bounce, and rotation about a distinct instantaneous centre.

In [ ]:
angled_anchors = mv.point(scenarios.angled_anchor_coordinates)               # [1] Point-
angled_attachments = mv.point(scenarios.angled_attachment_coordinates)       # [1] Point-
angled_constants = scenarios.angled_spring_constants                         # [1]

# Stiffness maps add across springs, so the third spring's dyad adds directly to the vertical pair.
angled_stiffness = spring_stiffness(angled_anchors, angled_attachments, angled_constants)  # [] Forque- <- Twist+
coupled_stiffness = unrestrained_stiffness + angled_stiffness                # [] Forque- <- Twist+
coupled_frequencies, coupled_modes = normal_modes(coupled_stiffness, inertia)  # [3] Scalar+, Twist+
print("Coupled frequencies (Hz):", np.round(coupled_frequencies.to_array(), 3))

coupled_anchors = mv.point(scenarios.coupled_anchor_coordinates)             # [3] Point-
coupled_attachments = mv.point(scenarios.coupled_attachment_coordinates)     # [3] Point-
mode_figure(corners, coupled_anchors, coupled_attachments, coupled_modes);

All three modes now carry positive frequencies and oblique displacements against the dotted equilibrium outline. Each panel tilts and shifts the plate simultaneously around a different point in the plane according to how the angled spring couples horizontal motion into bounce and rocking.

## Harmonic motion of the coupled modes

Released from rest in an isolated normal mode, the plate oscillates harmonically at that mode's natural frequency while keeping a fixed spatial twist profile. Streaming two full cycles of the lowest-frequency mode in shared physical time shows the three modes swinging at their distinct periods without exchanging energy.

In [ ]:
def mode_oscillation(frequencies: Scalar, mode_twists: Twist, n_frames: int) -> Iterator[Twist]:
    """Yields `n_frames` harmonic twist states (`[3] Twist+`) over two periods of the lowest nonzero frequency."""
    duration = 2 / frequencies[frequencies > 0].min()                        # [] Scalar+
    for fraction in np.linspace(0.0, 1.0, n_frames):
        phases = (2 * np.pi * frequencies * (fraction * duration)).cos()     # [3] Scalar+
        yield phases * mode_twists


n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms
oscillation = mode_oscillation(coupled_frequencies, coupled_modes, n_frames)
streamed_twists = show_states(
    oscillation, mode_frames, corners, coupled_anchors, coupled_attachments, coupled_modes,
    name="modes", duration_ms=frame_duration_ms, n_frames=n_frames,
)

Over the loop, the left panel completes two full swings at the lowest frequency while the middle and right panels oscillate faster at their higher natural frequencies, each plate tracing a fixed twist about its own centre of rotation.

In [ ]:
# checks
first_twist = coupled_modes[0] + coupled_modes[1]                            # [] Twist+
second_twist = coupled_modes[2]                                              # [] Twist+
potential_reciprocity = (first_twist & coupled_stiffness(second_twist)) - (second_twist & coupled_stiffness(first_twist))  # [] Scalar+
kinetic_reciprocity = (first_twist & inertia(second_twist)) - (second_twist & inertia(first_twist))                        # [] Scalar+
assert bool(potential_reciprocity.abs() < 1e-10)
assert bool(kinetic_reciprocity.abs() < 1e-10)

slide_forque = unrestrained_stiffness(unrestrained_modes[0])                 # [] Forque-
assert bool(unrestrained_frequencies[0].abs() < 1e-6)
assert bool((slide_forque.norm() + slide_forque.antinorm()) < 1e-10)

angular_rates_squared = (2 * np.pi * coupled_frequencies).squared()          # [3] Scalar+
modal_residuals = coupled_stiffness(coupled_modes) - angular_rates_squared * inertia(coupled_modes)  # [3] Forque-
off_diagonal_mass = coupled_modes[0] & inertia(coupled_modes[1])             # [] Scalar+
assert bool((coupled_frequencies > 0).all())
assert bool(((coupled_modes[:, None] & modal_residuals[None, :]).abs() < 1e-9).all())
assert bool(off_diagonal_mass.abs() < 1e-10)

start_error = corners[None, :].commutator((streamed_twists[0] - coupled_modes)[:, None]).antinorm()  # [3, 4] Scalar+
return_error = corners.commutator(streamed_twists[-1][0] - coupled_modes[0]).antinorm()              # [4] Scalar+
assert len(streamed_twists) == n_frames
assert bool((start_error < 1e-10).all())
assert bool((return_error < 1e-9).all())